# Week 04 — Function 08

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [Model and acquisition](#model-acquisition)
9. [Week 4 proposal](#week-4-proposal)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)

<a id="overview"></a>
## 1. Overview

This focused review mirrors the canonical Week 4 methodology for Function 08, with Weeks 1–3 observed and Week 4 proposed only.

<a id="objectives"></a>
## 2. Objectives

Validate the 8-dimensional evidence, assess the latest returned point, and reproduce the recorded GP-UCB proposal without look-ahead.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

Starter arrays come from `Week_01/Function_nn/03_Data`; exact returned pairs come from `Results/query_output_ledger.csv`. The Week 4 return is excluded because it was unknown when the proposal was selected.

<a id="environment-setup"></a>
## 4. Environment and setup

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
for candidate in (ROOT,*ROOT.parents):
    if (candidate/'Week_04'/'Function_08').is_dir(): ROOT=candidate; break
else: raise FileNotFoundError('Could not locate repository root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from Code.historical_function_review import analyse_historical_function

<a id="data-validation"></a>
## 5. Data validation

In [2]:
observations, summary, proposal, diagnostic_figure = analyse_historical_function(4, 8, ROOT)
input_columns=[f'x{i}' for i in range(1,9)]
inputs=observations[input_columns].to_numpy(float)
outputs=observations['objective'].to_numpy(float)
assert inputs.shape==(43,8) and outputs.shape==(43,)
assert np.isfinite(inputs).all() and np.isfinite(outputs).all()
assert np.all((inputs>=0)&(inputs<=1))
observations

,query,evidence,x1,x2,x3,x4,x5,x6,x7,x8,objective
0,1,starter,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,2,starter,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,3,starter,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,4,starter,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,5,starter,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,6,starter,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,7,starter,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,8,starter,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,9,starter,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,10,starter,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

All comparisons are descriptive and within-function; no causal, global-optimum, or cross-function ranking claim is made.

In [3]:
pd.Series({k:v for k,v in summary.items() if k!='proposal'}, name='verified evidence')

week                                                                             4
function                                                                         8
dimensions                                                                       8
starter_observations                                                            40
recorded_pairs                                                                   3
total_verified_observations                                                     43
best_query                                                                      42
best_input                       [0.16316, 0.184786, 0.152644, 0.083802, 0.9993...
best_output                                                               9.939904
latest_verified_query                                                           43
latest_verified_input            [0.088894, 0.525132, 0.030986, 0.992538, 0.870...
latest_verified_output                                                    8.963604
late

<a id="visual-eda"></a>
## 7. Visual EDA

Orange markers are returned Weeks 1–3 observations; the star is the verified incumbent. The Week 4 proposal is deliberately absent.

In [4]:
display(diagnostic_figure)
plt.close(diagnostic_figure)

<Figure size 1200x450 with 3 Axes>

<a id="model-acquisition"></a>
## 8. Model and acquisition

Method: **GP-UCB**. This adaptive policy is a heuristic chosen from the evidence available at the decision boundary; it is not a statistically controlled acquisition comparison.

<a id="week-4-proposal"></a>
## 9. Week 4 proposal

Proposed only: `[0.356502, 0.157574, 0.172997, 0.610561, 0.106477, 0.260297, 0.411421, 0.90474]`. Decision record: Chosen using evidence through Week 3, before the Week 4 return.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

In [5]:
candidate=np.asarray(proposal['query'],dtype=float)
assert proposal['status']=='proposed_only'
assert candidate.shape==(8,) and np.all((candidate>=0)&(candidate<=0.999999))
duplicate=bool(np.any(np.all(np.isclose(inputs,candidate,rtol=0,atol=5e-7),axis=1)))
assert duplicate==proposal['duplicates_observed_evidence']
assert summary['recorded_pairs']==3
portal='-'.join(f'{value:.6f}' for value in candidate)
assert all(len(part.split('.')[-1])==6 for part in portal.split('-'))
print('Function 08 Week 4 checks passed:', portal, 'duplicate:', duplicate)

Function 08 Week 4 checks passed: 0.356502-0.157574-0.172997-0.610561-0.106477-0.260297-0.411421-0.904740 duplicate: False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

The evidence boundary is locked at 43 verified observations. The Week 4 proposal remains unobserved until its authoritative return is appended at the next checkpoint.